In [29]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
config('spark.shuffle.useOldFetchProtocol','true'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [30]:
spark

In [31]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DateType, IntegerType, FloatType

In [32]:
hotel_schema = StructType([
    StructField("customer_id", LongType(), True),
    StructField("customer_name", StringType(), True),
    StructField("check_in_date", DateType(), True),
    StructField("check_out_date", DateType(), True),
    StructField("room_type", StringType(), True),
    StructField("price", FloatType(), True)
])

In [33]:
hotel_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(hotel_schema) \
.load("/public/trendytech/datasets/hotel_data.csv")

In [34]:
hotel_df

customer_id,customer_name,check_in_date,check_out_date,room_type,price
2,Jane Smith,2023-05-02,2023-05-06,Deluxe,600.0
3,Mark Johnson,2023-05-03,2023-05-08,Standard,450.0
4,Sarah Wilson,2023-05-04,2023-05-07,Executive,750.0
5,Emily Brown,2023-05-06,2023-05-09,Deluxe,550.0
6,Michael Davis,2023-05-07,2023-05-10,Standard,400.0
7,Samantha Thompson,2023-05-08,2023-05-12,Deluxe,600.0
8,William Lee,2023-05-10,2023-05-13,Standard,450.0
9,Amanda Harris,2023-05-11,2023-05-16,Executive,750.0
10,David Rodriguez,2023-05-12,2023-05-15,Deluxe,550.0
11,Linda Wilson,2023-05-14,2023-05-18,Standard,400.0


In [35]:
from pyspark.sql.functions import *

In [36]:
hotel_df.select(count("*").alias("row_count")).show()

+---------+
|row_count|
+---------+
|      106|
+---------+



In [37]:
summary_df = hotel_df.groupBy("room_type").agg(count("customer_id").alias("number_of_customer")).show()

+---------+------------------+
|room_type|number_of_customer|
+---------+------------------+
|Executive|                20|
|   Deluxe|                43|
| Standard|                43|
+---------+------------------+



## Grouping aggregation on hospital data

In [38]:
hospital_df = spark.read \
.format("csv") \
.option("header", "true") \
.option("inferSchema",True) \
.load("/public/trendytech/datasets/hospital.csv")

In [39]:
hospital_df.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- admission_date: string (nullable = true)
 |-- discharge_date: string (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- doctor_id: integer (nullable = true)
 |-- total_cost: double (nullable = true)



In [40]:
hospital_df.show()

+----------+--------------+--------------+-------------+---------+----------+
|patient_id|admission_date|discharge_date|    diagnosis|doctor_id|total_cost|
+----------+--------------+--------------+-------------+---------+----------+
|         1|    01-01-2022|    2022-01-10|    Pneumonia|      101|    5000.0|
|         2|    02-05-2022|    2022-02-09| Appendicitis|      102|    7000.0|
|         3|    03-12-2022|    2022-03-18|Fractured Arm|      103|    3500.0|
|         4|    04-02-2022|    2022-04-08| Heart Attack|      104|   15000.0|
|         5|    05-05-2022|    2022-05-07|    Influenza|      105|    2500.0|
|         6|    06-10-2022|    2022-06-15| Appendicitis|      106|    8000.0|
|         7|    07-20-2022|    2022-07-25|    Pneumonia|      107|    5500.0|
|         8|    08-25-2022|    2022-09-01| Heart Attack|      108|   20000.0|
|         9|    09-15-2022|    2022-09-22|Fractured Leg|      109|    6000.0|
|        10|    10-05-2022|    2022-10-10| Appendicitis|      11

In [41]:
summary_df = hospital_df.groupBy("diagnosis").agg(count("patient_id").alias("number_of_patients")).show()

+-------------+------------------+
|    diagnosis|number_of_patients|
+-------------+------------------+
| Heart Attack|                 5|
|Fractured Arm|                 3|
|Fractured Leg|                 2|
| Appendicitis|                 5|
|    Influenza|                 5|
|    Pneumonia|                 5|
+-------------+------------------+



In [56]:
### Converting string to dateformat

result_df1 = hospital_df.withColumn("admission_date", to_date("admission_date", "dd-mm-yyyy"))
result_df2 = result_df1.withColumn("discharge_date", to_date("discharge_date","yyyy-mm-dd"))

In [57]:
result_df2.show()

+----------+--------------+--------------+-------------+---------+----------+
|patient_id|admission_date|discharge_date|    diagnosis|doctor_id|total_cost|
+----------+--------------+--------------+-------------+---------+----------+
|         1|    2022-01-01|    2022-01-10|    Pneumonia|      101|    5000.0|
|         2|    2022-01-02|    2022-01-09| Appendicitis|      102|    7000.0|
|         3|    2022-01-03|    2022-01-18|Fractured Arm|      103|    3500.0|
|         4|    2022-01-04|    2022-01-08| Heart Attack|      104|   15000.0|
|         5|    2022-01-05|    2022-01-07|    Influenza|      105|    2500.0|
|         6|    2022-01-06|    2022-01-15| Appendicitis|      106|    8000.0|
|         7|    2022-01-07|    2022-01-25|    Pneumonia|      107|    5500.0|
|         8|    2022-01-08|    2022-01-01| Heart Attack|      108|   20000.0|
|         9|    2022-01-09|    2022-01-22|Fractured Leg|      109|    6000.0|
|        10|    2022-01-10|    2022-01-10| Appendicitis|      11

In [58]:
### Creating a new column called year which represents the year in which the patients are getting admitted
result_df3 = result_df2.withColumn("year",year("admission_date"))

In [59]:
result_df3.show()

+----------+--------------+--------------+-------------+---------+----------+----+
|patient_id|admission_date|discharge_date|    diagnosis|doctor_id|total_cost|year|
+----------+--------------+--------------+-------------+---------+----------+----+
|         1|    2022-01-01|    2022-01-10|    Pneumonia|      101|    5000.0|2022|
|         2|    2022-01-02|    2022-01-09| Appendicitis|      102|    7000.0|2022|
|         3|    2022-01-03|    2022-01-18|Fractured Arm|      103|    3500.0|2022|
|         4|    2022-01-04|    2022-01-08| Heart Attack|      104|   15000.0|2022|
|         5|    2022-01-05|    2022-01-07|    Influenza|      105|    2500.0|2022|
|         6|    2022-01-06|    2022-01-15| Appendicitis|      106|    8000.0|2022|
|         7|    2022-01-07|    2022-01-25|    Pneumonia|      107|    5500.0|2022|
|         8|    2022-01-08|    2022-01-01| Heart Attack|      108|   20000.0|2022|
|         9|    2022-01-09|    2022-01-22|Fractured Leg|      109|    6000.0|2022|
|   

find out number of patients diagnosed in each year categorized by different categories

In [60]:
summary_df = result_df3.groupBy("year","diagnosis").agg(count("patient_id").alias("totaL_patient")).sort(desc("year"))
summary_df.show()

+----+-------------+-------------+
|year|    diagnosis|totaL_patient|
+----+-------------+-------------+
|2024|    Influenza|            1|
|2023|    Influenza|            2|
|2023| Appendicitis|            2|
|2023|    Pneumonia|            2|
|2023| Heart Attack|            3|
|2023|Fractured Leg|            1|
|2023|Fractured Arm|            2|
|2022| Appendicitis|            3|
|2022|Fractured Leg|            1|
|2022|    Pneumonia|            3|
|2022|    Influenza|            2|
|2022| Heart Attack|            2|
|2022|Fractured Arm|            1|
+----+-------------+-------------+



In [64]:
# creating pivottable based on data and replacing the null values by 0
summary_df.groupBy("diagnosis").pivot("year").count().na.fill(0).show()

+-------------+----+----+----+
|    diagnosis|2022|2023|2024|
+-------------+----+----+----+
| Heart Attack|   1|   1|   0|
|Fractured Arm|   1|   1|   0|
|Fractured Leg|   1|   1|   0|
| Appendicitis|   1|   1|   0|
|    Influenza|   1|   1|   1|
|    Pneumonia|   1|   1|   0|
+-------------+----+----+----+



In [67]:
## lets find the total number of patients for each category
from pyspark.sql import Window
from pyspark.sql.functions import desc

In [69]:
my_window = Window.partitionBy("diagnosis")

In [72]:
summary_df2 = result_df3.withColumn("total_patient_each_category", count("patient_id").over(my_window))

In [73]:
summary_df2.show()

+----------+--------------+--------------+-------------+---------+----------+----+---------------------------+
|patient_id|admission_date|discharge_date|    diagnosis|doctor_id|total_cost|year|total_patient_each_category|
+----------+--------------+--------------+-------------+---------+----------+----+---------------------------+
|         4|    2022-01-04|    2022-01-08| Heart Attack|      104|   15000.0|2022|                          5|
|         8|    2022-01-08|    2022-01-01| Heart Attack|      108|   20000.0|2022|                          5|
|        13|    2023-01-01|    2023-01-09| Heart Attack|      113|   18000.0|2023|                          5|
|        17|    2023-01-05|    2023-01-11| Heart Attack|      117|   16000.0|2023|                          5|
|        22|    2023-01-10|    2023-01-19| Heart Attack|      122|   21000.0|2023|                          5|
|         3|    2022-01-03|    2022-01-18|Fractured Arm|      103|    3500.0|2022|                          3|
|

In [28]:
spark.stop()